# Check data quality

Inspect the EDF files and channel metadata before extracting features.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)

In [2]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "capslpdb" / "recordings.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook inside the project directory.")
DATA_DIR = PROJECT_ROOT / "data" / "capslpdb"
REPORT_DIR = PROJECT_ROOT / "outputs" / "01_data_quality"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
EPOCH_SECONDS = 30
EEG_BANDS = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0)}
CHANNEL_CANDIDATES = {
    "eeg": [("C4-A1",), ("C4A1",), ("C4", "A1"), ("C3-A2",), ("C3A2",)],
    "eog": [("ROC-LOC",), ("LOC-ROC",), ("ROC", "LOC"), ("EOG-R", "EOG-L")],
    "emg": [("EMG1-EMG2",), ("EMG-EMG",), ("EMG1", "EMG2")],
}
CHANNEL_OVERRIDES = {
    "n4": {"eog": ("EOG dx", "EOG sin"), "emg": ("milo",)},
    "n6": {"eog": ("ROC-A2", "LOC-A1")},
    "n7": {"eog": ("ROC-A2", "LOC-A1")},
    "n8": {"emg": ("milo",)},
    "n9": {"eog": ("ROC-A2", "LOC-A1"), "emg": ("EMG",)},
    "n12": {"eog": ("ROC / A1", "LOC / A2"), "emg": (2, 3)},
    "n13": {"emg": ("CHIN1", "CHIN2")},
    "n14": {"emg": ("CHIN1", "CHIN2")},
    "n15": {"emg": ("CHIN1", "CHIN2")},
}

In [3]:
mapping = pd.read_csv(DATA_DIR / "recordings.csv", dtype=str)
required = {"record_id", "class_name", "original_group", "edf_path"}
if not required.issubset(mapping.columns):
    raise ValueError("Required mapping columns are missing.")
if mapping[list(required)].isna().any().any() or not mapping["record_id"].str.fullmatch(r"[a-z]+[0-9]+").all() or mapping["record_id"].duplicated().any():
    raise ValueError("The recordings mapping is incomplete or contains duplicate identifiers.")
if not mapping["class_name"].isin(["healthy", "rbd", "other"]).all():
    raise ValueError("Unexpected class labels.")
if mapping.empty:
    raise ValueError("No recordings were selected.")
mapping = mapping.sort_values("record_id").reset_index(drop=True)

def recording_path(relative_path):
    path = (DATA_DIR / relative_path).resolve()
    if not path.is_relative_to(DATA_DIR.resolve()):
        raise ValueError("The recording path leaves the dataset directory.")
    return path

display(mapping.groupby("class_name").size().rename("recordings").to_frame())

,recordings
class_name,
healthy,16
other,70
rbd,22


In [4]:
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
import importlib
import edf_quality
importlib.reload(edf_quality)
from edf_quality import inspect_recordings, resolve_channel as _resolve_channel

def resolve_channel(edf, record_id, modality):
    return _resolve_channel(edf, record_id, modality, ACTIVE_OVERRIDES, CHANNEL_CANDIDATES, EPOCH_SECONDS, EEG_BANDS)

## Inspect recordings

In [5]:
def flag_missing_units(headers, plans, inventory):
    for record_id, plan in plans.items():
        edf = headers[record_id]
        missing = [edf.labels[index] for selection in plan.values() for index in selection["indices"]
                   if edf.units[index].strip().lower().replace("µ", "u").replace("μ", "u") not in {"uv", "mv", "v"}]
        if missing:
            rows = inventory["record_id"].eq(record_id)
            inventory.loc[rows, "status"] = "review_units"
            inventory.loc[rows, "reason"] = "Missing or unsupported unit for: " + ", ".join(missing)

def resolve_automatic(edf, record_id, modality):
    return _resolve_channel(edf, record_id, modality, {}, CHANNEL_CANDIDATES, EPOCH_SECONDS, EEG_BANDS, require_units=False)

automatic_headers, automatic_plans, automatic_inventory = inspect_recordings(
    mapping, recording_path, CHANNEL_CANDIDATES, resolve_automatic)
flag_missing_units(automatic_headers, automatic_plans, automatic_inventory)
display(pd.crosstab(automatic_inventory["class_name"], automatic_inventory["status"]))

issue_rows = []
for row in automatic_inventory.loc[automatic_inventory["status"].ne("ready")].itertuples():
    available = row.record_id in automatic_headers
    missing = []
    if available:
        edf = automatic_headers[row.record_id]
        missing = [label for label, unit in zip(edf.labels, edf.units) if not unit.strip()]
    issue = "Missing unit for: " + ", ".join(missing) if missing else row.reason if not available else ""
    issue_rows.append({"record_id": row.record_id, "class_name": row.class_name,
                       "eeg_auto_match": bool(row.eeg_channels) if available else None,
                       "eog_auto_match": bool(row.eog_channels) if available else None,
                       "emg_auto_match": bool(row.emg_channels) if available else None,
                       "additional_issues": issue})
issues = pd.DataFrame(issue_rows)
display(issues)

Inspect EDF headers: 100%|██████████| 108/108 [00:00<00:00, 1328.21record/s]


status,invalid_recording,ready,review_channels
class_name,,,
healthy,0,6,10
other,1,69,0
rbd,0,22,0


,record_id,class_name,eeg_auto_match,eog_auto_match,emg_auto_match,additional_issues
0,brux1,other,None,None,None,File size does not match the number of data blocks in the EDF header.
1,n12,healthy,True,False,False,
2,n13,healthy,True,True,False,"Missing unit for: LOC, ROC, CHIN1, CHIN2, F3A2, F4A1, C3A2, C4A1, O1A2, O2A1, ECG"
3,n14,healthy,True,True,False,"Missing unit for: LOC, ROC, CHIN1, CHIN2, F3A2, F4A1, C3A2, C4A1, O1A2, O2A1, ECG"
4,n15,healthy,True,True,False,"Missing unit for: EOG-L, EOG-R, CHIN1, CHIN2, F3A2, F4A1, C3A2, C4A1, O1A2, O2A1, EKG"
5,n16,healthy,True,False,False,
6,n4,healthy,True,False,False,
7,n6,healthy,True,False,True,
8,n7,healthy,True,False,True,
9,n8,healthy,True,True,False,


**Note:** The brux1 header specifies 32,340 data blocks, but the file contains only 14,341 complete blocks and 8,704 bytes of another block and is therefore excluded.

## Select channels automatically

Select channels using the first matching general rule and show them in green.

In [6]:
EXCLUDED_RECORDINGS = {"brux1": "Incomplete EDF data blocks and inconsistent recording length"}

def describe_selection(edf, record_id, modality, resolver):
    try:
        selection = resolver(edf, record_id, modality)
    except ValueError:
        return ""
    if selection is None:
        return ""
    return " minus ".join(f"{label} [index {index}]" for label, index in zip(selection["labels"], selection["indices"]))

automatic_review = automatic_inventory.loc[
    ~automatic_inventory["record_id"].isin(EXCLUDED_RECORDINGS) & automatic_inventory["status"].ne("ready"),
    ["record_id", "eeg_channels", "eog_channels", "emg_channels", "status"]].copy()
for row in automatic_review.itertuples():
    for modality in CHANNEL_CANDIDATES:
        automatic_review.loc[row.Index, f"{modality}_channels"] = describe_selection(
            automatic_headers[row.record_id], row.record_id, modality, resolve_automatic)
def highlight_selection(value):
    if not value:
        return ""
    color = "#ffe4c4" if value.endswith(" (manual)") else "#dff1df"
    return f"background-color: {color}; color: #202020"

def show_selections(table):
    columns = [f"{modality}_channels" for modality in CHANNEL_CANDIDATES]
    styled = table.fillna("").reset_index(drop=True).style.map(highlight_selection, subset=columns)
    display(styled.set_properties(**{"border-bottom": "1px solid #777", "border-right": "1px solid #777"}))

show_selections(automatic_review)

,record_id,eeg_channels,eog_channels,emg_channels,status
0,n12,C4-A1 [index 5],,,review_channels
1,n13,C4A1 [index 7],ROC [index 1] minus LOC [index 0],,review_channels
2,n14,C4A1 [index 7],ROC [index 1] minus LOC [index 0],,review_channels
3,n15,C4A1 [index 7],EOG-R [index 1] minus EOG-L [index 0],,review_channels
4,n16,C4-A1 [index 4],,,review_channels
5,n4,C4-A1 [index 5],,,review_channels
6,n6,C3-A2 [index 3],,EMG-EMG [index 2],review_channels
7,n7,C3-A2 [index 3],,EMG-EMG [index 2],review_channels
8,n8,C3-A2 [index 0],EOG-R [index 4] minus EOG-L [index 3],,review_channels
9,n9,C3-A2 [index 3],,,review_channels


## Apply channel overrides

Apply predefined rules where no channel was found automatically and show these selections in orange.

In [7]:
automatic_by_record = automatic_inventory.set_index("record_id")
ACTIVE_OVERRIDES = {}
for record_id, overrides in CHANNEL_OVERRIDES.items():
    if record_id not in automatic_by_record.index or record_id in EXCLUDED_RECORDINGS:
        continue
    missing = {modality: channels for modality, channels in overrides.items()
               if pd.isna(automatic_by_record.loc[record_id, f"{modality}_channels"]) or
               not str(automatic_by_record.loc[record_id, f"{modality}_channels"]).strip()}
    if missing:
        ACTIVE_OVERRIDES[record_id] = missing

def resolve_configured_names(edf, record_id, modality):
    return _resolve_channel(edf, record_id, modality, ACTIVE_OVERRIDES, CHANNEL_CANDIDATES, EPOCH_SECONDS, EEG_BANDS, require_units=False)

channel_mapping = mapping[~mapping["record_id"].isin(EXCLUDED_RECORDINGS)]
name_headers, name_plans, name_inventory = inspect_recordings(
    channel_mapping, recording_path, CHANNEL_CANDIDATES, resolve_configured_names)
flag_missing_units(name_headers, name_plans, name_inventory)

affected = automatic_inventory.loc[automatic_inventory["status"].ne("ready"), "record_id"]
configured_review = name_inventory.loc[
    name_inventory["record_id"].isin(affected),
    ["record_id", "eeg_channels", "eog_channels", "emg_channels", "status"]].copy()
for row in configured_review.itertuples():
    for modality in CHANNEL_CANDIDATES:
        selection = describe_selection(name_headers[row.record_id], row.record_id, modality, resolve_configured_names)
        if selection and modality in ACTIVE_OVERRIDES.get(row.record_id, {}):
            selection += " (manual)"
        configured_review.loc[row.Index, f"{modality}_channels"] = selection
show_selections(configured_review)

Inspect EDF headers: 100%|██████████| 107/107 [00:00<00:00, 1641.20record/s]


,record_id,eeg_channels,eog_channels,emg_channels,status
0,n12,C4-A1 [index 5],ROC / A1 [index 1] minus LOC / A2 [index 0] (manual),CHIN [index 2] minus CHIN [index 3] (manual),ready
1,n13,C4A1 [index 7],ROC [index 1] minus LOC [index 0],CHIN1 [index 2] minus CHIN2 [index 3] (manual),review_units
2,n14,C4A1 [index 7],ROC [index 1] minus LOC [index 0],CHIN1 [index 2] minus CHIN2 [index 3] (manual),review_units
3,n15,C4A1 [index 7],EOG-R [index 1] minus EOG-L [index 0],CHIN1 [index 2] minus CHIN2 [index 3] (manual),review_units
4,n16,C4-A1 [index 4],,,review_channels
5,n4,C4-A1 [index 5],EOG dx [index 0] minus EOG sin [index 7] (manual),milo [index 6] (manual),ready
6,n6,C3-A2 [index 3],ROC-A2 [index 1] minus LOC-A1 [index 0] (manual),EMG-EMG [index 2],ready
7,n7,C3-A2 [index 3],ROC-A2 [index 1] minus LOC-A1 [index 0] (manual),EMG-EMG [index 2],ready
8,n8,C3-A2 [index 0],EOG-R [index 4] minus EOG-L [index 3],milo [index 12] (manual),ready
9,n9,C3-A2 [index 3],ROC-A2 [index 1] minus LOC-A1 [index 0] (manual),EMG [index 2] (manual),ready


**Note:** The n16 recording contains only five EEG channels (Fp2-F4, F4-C4, C4-P4, P4-O2 and C4-A1), and has no EOG or EMG channels, so it cannot be used for the multimodal analysis.

In [8]:
EXCLUDED_RECORDINGS["n16"] = "No EOG or EMG channels in the EDF header"

headers, configured_plans, inventory = inspect_recordings(
    mapping, recording_path, CHANNEL_CANDIDATES, resolve_channel)
plans = {record_id: plan for record_id, plan in configured_plans.items() if record_id not in EXCLUDED_RECORDINGS}
inventory["excluded_from_analysis"] = inventory["record_id"].isin(EXCLUDED_RECORDINGS)

Inspect EDF headers: 100%|██████████| 108/108 [00:00<00:00, 1782.11record/s]


## Compare missing units

Compare missing units with the units of channels in the reference recordings.

In [9]:
channel_rows = []
for record_id, edf in headers.items():
    for index, label in enumerate(edf.labels):
        channel_rows.append({"record_id": record_id, "channel_index": index,
                             "channel_label": label, "unit": edf.units[index]})
channels = pd.DataFrame(channel_rows)

def comparison_key(label, record_id=None, index=None):
    name = "".join(char for char in label.upper() if char.isalnum())
    if name in {"CHIN1CHIN2", "CHIN0CHIN1", "EMG1EMG2"}:
        return "CHIN_DIFFERENCE"
    if name == "CHIN" and record_id == "n12" and index in (2, 3):
        return f"CHIN{index - 1}"
    if name in {"ECG", "EKG"}:
        return "ECG"
    return name

targets = channels[channels["record_id"].isin(["n13", "n14", "n15"]) & channels["unit"].fillna("").str.strip().eq("")]
references = channels[channels["record_id"].isin(plans) & channels["unit"].fillna("").str.strip().ne("")].copy()
references["comparison_key"] = references.apply(lambda row: comparison_key(row["channel_label"], row["record_id"], row["channel_index"]), axis=1)
references["reference_display"] = references.apply(lambda row: f"CHIN-{row['channel_index'] - 2} (n12)" if row["record_id"] == "n12" and row["channel_label"] == "CHIN" else row["channel_label"], axis=1)
comparison_rows = []
for label in sorted(targets["channel_label"].unique()):
    key = comparison_key(label)
    reference_keys = [key, "CHIN_DIFFERENCE"] if key in {"CHIN1", "CHIN2"} else [key]
    matches = references[references["comparison_key"].isin(reference_keys)]
    if matches.empty:
        comparison_rows.append({"missing_channel": label, "reference_channel": "No similar channel found", "unit": "", "recordings": 0})
    else:
        counts = matches.groupby(["reference_display", "unit"])["record_id"].nunique()
        for (reference, unit), count in counts.items():
            comparison_rows.append({"missing_channel": label, "reference_channel": reference, "unit": unit, "recordings": count})
unit_comparison = pd.DataFrame(comparison_rows)
print("Units in usable recordings for channels with missing units:")
display(unit_comparison)

Units in usable recordings for channels with missing units:


,missing_channel,reference_channel,unit,recordings
0,C3A2,C3-A2,mV,1
1,C3A2,C3-A2,uV,4
2,C4A1,C4-A1,uV,97
3,CHIN1,CHIN-0 (n12),uV,1
4,CHIN1,EMG1-EMG2,mV,7
5,CHIN1,EMG1-EMG2,uV,88
6,CHIN2,CHIN-1 (n12),uV,1
7,CHIN2,EMG1-EMG2,mV,7
8,CHIN2,EMG1-EMG2,uV,88
9,ECG,ECG,uV,3


## Compare signal amplitudes

Compare up to 300 randomly selected 30-second windows per recording using a fixed seed and skip windows with at least 1% samples at the saturation limits found in the channel.

In [10]:
import numpy as np

WINDOW_SECONDS = 30
WINDOWS_PER_RECORDING = 300
target_ids = ["n13", "n14", "n15"]

def comparison_name(label, record_id=None, index=None):
    name = "".join(char for char in label.upper() if char.isalnum())
    if name in {"CHIN1CHIN2", "CHIN0CHIN1", "EMG1EMG2"}:
        return "CHIN_DIFFERENCE"
    if name == "CHIN" and record_id == "n12" and index in (2, 3):
        return f"CHIN{index - 1}"
    return name

target_channels = [(record_id, index, label, comparison_name(label, record_id, index))
                   for record_id in target_ids if record_id in headers
                   for index, label in enumerate(headers[record_id].labels)]
comparison_names = {name for _, _, _, name in target_channels}
comparison_names.add("CHIN_DIFFERENCE")

def select_windows(complete):
    """Choose one random window per section of the recording."""
    count = min(WINDOWS_PER_RECORDING, complete)
    boundaries = np.linspace(0, complete, count + 1, dtype=int)
    rng = np.random.default_rng(RANDOM_SEED)
    return np.array([rng.integers(start, stop) for start, stop in zip(boundaries[:-1], boundaries[1:])])

def amplitude_sample(edf, index):
    rate = edf.sfreq(index)
    samples = round(WINDOW_SECONDS * rate)
    complete = int(edf.duration // WINDOW_SECONDS)
    if not complete or not np.isclose(samples, WINDOW_SECONDS * rate):
        return None, 0
    starts = select_windows(complete)
    digital_low, digital_high, _ = edf.digital_limits(index)
    digital_span = edf.digital_max[index] - edf.digital_min[index]
    if digital_span == 0 or edf.physical_max[index] == edf.physical_min[index]:
        return None, 0
    amplitudes = []
    for window in starts:
        start = int(window) * samples
        stop = start + samples
        digital = edf.read_digital(index, start, stop)
        if len(digital) != samples or np.mean((digital <= digital_low) | (digital >= digital_high)) >= 0.01 or np.mean(np.diff(digital) == 0) >= 0.995:
            continue
        scale = abs((edf.physical_max[index] - edf.physical_min[index]) / digital_span)
        amplitudes.append(float(np.ptp(digital.astype(float)) * scale))
    return (float(np.median(amplitudes)), len(amplitudes)) if amplitudes else (None, 0)

samples = []
for record_id in [*target_ids, *plans]:
    if record_id not in headers:
        continue
    edf = headers[record_id]
    with edf:
        for index, label in enumerate(edf.labels):
            name = comparison_name(label, record_id, index)
            if name not in comparison_names:
                continue
            unit = edf.units[index].strip().lower().replace("µ", "u").replace("μ", "u")
            factor = {"uv": 1, "mv": 1000, "v": 1_000_000}.get(unit)
            if record_id not in target_ids and factor is None:
                continue
            amplitude, count = amplitude_sample(edf, index)
            samples.append({"record_id": record_id, "channel": label, "comparison_name": name,
                            "unit": unit, "amplitude": amplitude, "windows": count, "factor": factor})

sample_table = pd.DataFrame(samples)
results = []
for record_id, index, label, name in target_channels:
    current = sample_table.loc[(sample_table["record_id"] == record_id) & (sample_table["channel"] == label)]
    if current.empty:
        continue
    current = current.iloc[0]
    groups = [("n12 CHIN-0" if name == "CHIN1" else "n12 CHIN-1", name), ("EMG1-EMG2", "CHIN_DIFFERENCE")] if name in {"CHIN1", "CHIN2"} else [(label, name)]
    for reference_label, reference_key in groups:
        references = sample_table.loc[~sample_table["record_id"].isin(target_ids) &
                                      sample_table["comparison_name"].eq(reference_key) & sample_table["amplitude"].notna()].copy()
        references["amplitude_uv"] = references["amplitude"] * references["factor"]
        per_record = references.groupby("record_id")["amplitude_uv"].median()
        results.append({"record_id": record_id, "channel": label, "reference_channel": reference_label,
            "valid_windows": int(current["windows"]), "if_uV": current["amplitude"],
            "if_mV_in_uV": current["amplitude"] * 1000 if pd.notna(current["amplitude"]) else np.nan,
            "reference_recordings": len(per_record), "reference_median_uV": per_record.median(),
            "reference_p10_uV": per_record.quantile(0.10), "reference_p90_uV": per_record.quantile(0.90)})

amplitude_comparison = pd.DataFrame(results)

In [11]:
def recommend_unit(row):
    if row["reference_recordings"] == 0 or pd.isna(row["reference_median_uV"]):
        return "No reference"
    if row["channel"].upper().startswith("CHIN"):
        return "Check CHIN"
    if row["valid_windows"] < 20 or pd.isna(row["if_uV"]):
        return "Insufficient windows"
    target = row["reference_median_uV"]
    if target <= 0 or row["if_uV"] <= 0:
        return "No recommendation"
    ratios = {"uV": row["if_uV"] / target, "mV": row["if_mV_in_uV"] / target}
    distances = {unit: abs(np.log10(ratio)) for unit, ratio in ratios.items()}
    best = min(distances, key=distances.get)
    other = "mV" if best == "uV" else "uV"
    if distances[other] - distances[best] < 1:
        return "No clear recommendation"
    if distances[best] > 1:
        return f"{best} (far from reference)"
    return f"{best} (tentative)"

def highlight_review(row):
    color = "#fff0d6" if row["reference_recordings"] == 0 else "#e6f1ff" if row["channel"].upper().startswith("CHIN") else ""
    return [f"background-color: {color}; color: #202020" if color else ""] * len(row)

amplitude_comparison["recommended_unit"] = amplitude_comparison.apply(recommend_unit, axis=1)
styled_comparison = (amplitude_comparison.style
    .set_table_attributes('class="dataframe"')
    .format(precision=2, na_rep="-")
    .apply(highlight_review, axis=1)
    .set_properties(**{"border-bottom": "1px solid #777", "border-right": "1px solid #777"})
    .set_table_styles([{"selector": "th", "props": [("border-bottom", "1px solid #777"), ("border-right", "1px solid #777")]}]))
display(styled_comparison)

,record_id,channel,reference_channel,valid_windows,if_uV,if_mV_in_uV,reference_recordings,reference_median_uV,reference_p10_uV,reference_p90_uV,recommended_unit
0,n13,LOC,LOC,256,141.59,141587.24,2,185.71,180.31,191.12,uV (tentative)
1,n13,ROC,ROC,264,107.07,107068.67,2,178.58,172.76,184.40,uV (tentative)
2,n13,CHIN1,n12 CHIN-0,277,287.00,287000.00,1,13.17,13.17,13.17,Check CHIN
3,n13,CHIN1,EMG1-EMG2,277,287.00,287000.00,95,33.50,15.97,129.89,Check CHIN
4,n13,CHIN2,n12 CHIN-1,277,298.00,298000.00,1,13.17,13.17,13.17,Check CHIN
5,n13,CHIN2,EMG1-EMG2,277,298.00,298000.00,95,33.50,15.97,129.89,Check CHIN
6,n13,F3A2,F3A2,260,242.55,242546.43,0,-,-,-,No reference
7,n13,F4A1,F4A1,268,231.70,231702.10,0,-,-,-,No reference
8,n13,C3A2,C3A2,286,214.14,214137.34,5,130.90,111.28,191.71,uV (tentative)
9,n13,C4A1,C4A1,284,207.88,207875.12,97,148.41,105.64,190.88,uV (tentative)


## Save EDF copies with assumed units

The CHIN difference is read later as a bipolar montage of the two CHIN channels, so a clipped sample in either channel also marks the difference as clipped.

In [12]:
import shutil
import tempfile

ASSUMED_UNITS = {
    "n13": {"LOC": "uV", "ROC": "uV", "CHIN1": "uV", "CHIN2": "uV", "C3A2": "uV", "C4A1": "uV", "O1A2": "uV", "O2A1": "uV", "ECG": "uV"},
    "n14": {"LOC": "uV", "ROC": "uV", "CHIN1": "uV", "CHIN2": "uV", "C3A2": "uV", "C4A1": "uV", "O1A2": "uV", "O2A1": "uV", "ECG": "uV"},
    "n15": {"EOG-L": "uV", "EOG-R": "uV", "CHIN1": "uV", "CHIN2": "uV", "C3A2": "uV", "C4A1": "uV", "O1A2": "uV", "O2A1": "uV", "EKG": "uV"},
}

def save_unit_copy(edf, updates, destination):
    if destination.exists() or destination.resolve() == edf.path.resolve():
        raise ValueError(f"Choose a new output file: {destination}")
    selections = []
    for label, unit in updates.items():
        matches = [index for index, actual in enumerate(edf.labels) if actual == label]
        if len(matches) != 1:
            raise ValueError(f"Channel name must occur exactly once: {label}")
        index = matches[0]
        if edf.units[index].strip():
            raise ValueError(f"The unit is already present for {label}")
        if unit not in {"uV", "mV", "V"}:
            raise ValueError(f"Unsupported unit for {label}: {unit}")
        selections.append((index, unit))
    with edf.path.open("rb") as source:
        header = bytearray(source.read(edf.header_bytes))
        if len(header) != edf.header_bytes:
            raise ValueError("The EDF header is incomplete.")
        for index, unit in selections:
            offset = 256 + edf.channel_count * 96 + index * 8
            header[offset:offset + 8] = unit.encode("ascii").ljust(8, b" ")
        with destination.open("xb") as output:
            output.write(header)
            shutil.copyfileobj(source, output, length=1024 * 1024)

copy_rows = []
for record_id, units in ASSUMED_UNITS.items():
    original = headers[record_id]
    destination = original.path.with_name(f"{record_id}_prepared.edf")
    if destination.resolve() == original.path.resolve():
        raise ValueError("The original EDF file cannot be overwritten.")
    with tempfile.TemporaryDirectory(dir=destination.parent) as folder:
        temporary = Path(folder) / destination.name
        save_unit_copy(original, units, temporary)
        temporary.replace(destination)
    copy_rows.append({"record_id": record_id, "assumed_units": ", ".join(units), "copy": destination.name})
display(pd.DataFrame(copy_rows))

,record_id,assumed_units,copy
0,n13,"LOC, ROC, CHIN1, CHIN2, C3A2, C4A1, O1A2, O2A1, ECG",n13_prepared.edf
1,n14,"LOC, ROC, CHIN1, CHIN2, C3A2, C4A1, O1A2, O2A1, ECG",n14_prepared.edf
2,n15,"EOG-L, EOG-R, CHIN1, CHIN2, C3A2, C4A1, O1A2, O2A1, EKG",n15_prepared.edf


## Check final channel selections

Check the selected files and channels and save the results to a CSV file.

In [13]:
from edf_quality import selection_review, uncalibrated

prepared_files = {row["record_id"]: headers[row["record_id"]].path.with_name(row["copy"]) for row in copy_rows}
final_mapping = mapping.copy()
final_overrides = dict(ACTIVE_OVERRIDES)
for record_id, path in prepared_files.items():
    final_mapping.loc[final_mapping["record_id"].eq(record_id), "edf_path"] = path.relative_to(DATA_DIR).as_posix()

def resolve_final(edf, record_id, modality):
    return _resolve_channel(edf, record_id, modality, final_overrides, CHANNEL_CANDIDATES, EPOCH_SECONDS, EEG_BANDS, require_units=False)

final_headers, final_plans, final_inventory = inspect_recordings(
    final_mapping, recording_path, CHANNEL_CANDIDATES, resolve_final)
selection_rows = []
for record in mapping.itertuples():
    record_id = record.record_id
    original = recording_path(record.edf_path)
    selected_file = prepared_files.get(record_id, original)
    row = {"record_id": record_id, "class_name": record.class_name,
           "original_edf": record.edf_path, "selected_edf": selected_file.relative_to(DATA_DIR).as_posix(),
           "status": "selected", "issues": []}
    if record_id in EXCLUDED_RECORDINGS:
        row["status"] = "excluded"
        row["issues"].append(EXCLUDED_RECORDINGS[record_id])
        selection_rows.append(row)
        continue
    try:
        if record_id not in final_headers:
            reason = final_inventory.loc[final_inventory["record_id"].eq(record_id), "reason"].iloc[0]
            raise ValueError(reason)
        edf = final_headers[record_id]
        for modality in CHANNEL_CANDIDATES:
            selection = _resolve_channel(edf, record_id, modality, final_overrides, CHANNEL_CANDIDATES, EPOCH_SECONDS, EEG_BANDS)
            if selection is None:
                row["status"] = "review"
                row["issues"].append(f"No {modality.upper()} channel selected")
                continue
            indices = selection["indices"]
            row[f"{modality}_indices"] = ",".join(str(index) for index in indices)
            row[f"{modality}_channels"] = " minus ".join(f"{edf.labels[index]} [index {index}]" for index in indices)
            row[f"{modality}_units"] = ", ".join(edf.units[index] for index in indices)
            row[f"{modality}_method"] = "manual" if modality in final_overrides.get(record_id, {}) else "automatic"
            missing_calibration = [edf.labels[index] for index in indices if uncalibrated(edf, index)]
            row[f"{modality}_calibrated"] = not missing_calibration
            if missing_calibration:
                row["issues"].append(f"{modality.upper()}: No calibration in the header for {', '.join(missing_calibration)}. "
                                     "Physical values equal digital values.")
            review = selection_review(modality, selection)
            if review:
                row["issues"].append(f"{modality.upper()}: {review}")
        if record_id in ASSUMED_UNITS:
            row["issues"].append("Some units are assumed")
            missing = [label for label, unit in zip(edf.labels, edf.units) if not unit.strip()]
            if missing:
                row["issues"].append("Missing units for: " + ", ".join(missing))
    except (OSError, ValueError, OverflowError) as error:
        row["status"] = "review"
        row["issues"].append(str(error))
    selection_rows.append(row)

selection_report = pd.DataFrame(selection_rows).fillna("")
selection_report["issues"] = selection_report["issues"].map(lambda issues: " | ".join(issues))
report_path = REPORT_DIR / "channel_selections.csv"
selection_report.to_csv(report_path, index=False)

flag_missing_units(final_headers, final_plans, final_inventory)
display(pd.crosstab(final_inventory["class_name"], final_inventory["status"]))

final_issues = []
for row in final_inventory.itertuples():
    available = row.record_id in final_headers
    missing = []
    if available and row.record_id in prepared_files:
        edf = final_headers[row.record_id]
        missing = [label for label, unit in zip(edf.labels, edf.units) if not unit.strip()]
    if row.status == "ready" and not missing:
        continue
    issue = "Missing unit for: " + ", ".join(missing) if missing else row.reason.replace("; ", ", ")
    final_issues.append({"record_id": row.record_id, "class_name": row.class_name,
                         "eeg_available": bool(row.eeg_channels) if available else None,
                         "eog_available": bool(row.eog_channels) if available else None,
                         "emg_available": bool(row.emg_channels) if available else None,
                         "additional_issues": issue,
                         "excluded": row.record_id in EXCLUDED_RECORDINGS})
display(pd.DataFrame(final_issues))
print(f"Channel selection CSV: {report_path.relative_to(PROJECT_ROOT).as_posix()}")

Inspect EDF headers: 100%|██████████| 108/108 [00:00<00:00, 1857.02record/s]


status,invalid_recording,ready,review_channels
class_name,,,
healthy,0,15,1
other,1,69,0
rbd,0,22,0


,record_id,class_name,eeg_available,eog_available,emg_available,additional_issues,excluded
0,brux1,other,None,None,None,File size does not match the number of data blocks in the EDF header.,True
1,n13,healthy,True,True,True,"Missing unit for: F3A2, F4A1",False
2,n14,healthy,True,True,True,"Missing unit for: F3A2, F4A1",False
3,n15,healthy,True,True,True,"Missing unit for: F3A2, F4A1",False
4,n16,healthy,True,False,False,"EOG: No channel selected, EMG: No channel selected",True


Channel selection CSV: outputs/01_data_quality/channel_selections.csv
